<!-- # Job Accessibility -->

In [ ]:
import datetime
from pathlib import Path

import pandas as pd
import geopandas as gpd
import r5py


def find_project_root(marker: str = "pyproject.toml") -> Path:
    """Walk up from the current working directory to find the repo root.

    Notebooks run with their working directory set to ``notebooks/``, so
    this lets them resolve ``data/`` paths without hardcoding ``../data``.
    """
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data"

OSM = DATA_PATH / "uruguay-260624.osm.pbf"
EMPLOYMENT = DATA_PATH / "URY_montevideo_employment_predictions_relative.gpkg"
FEEDS = [
    DATA_PATH / "GTFS" / "google_transit.zip",
    DATA_PATH / "GTFS" / "gtfs_20260326.zip",
]
OUT = DATA_PATH / "opportunities"
OUT.mkdir(parents=True, exist_ok=True)

LOCAL_CRS = "EPSG:32721"
WGS84_CRS = "EPSG:4326"

DEPARTURE = datetime.datetime(2026, 3, 25, 8, 0)

employment_grid = gpd.read_file(EMPLOYMENT)
employment = (
    employment_grid.copy()
    .assign(
        geometry=lambda df: df.geometry.to_crs(LOCAL_CRS).centroid.to_crs(WGS84_CRS)
    )
    .rename(columns={"grid_id": "id"})
)

network = r5py.TransportNetwork(OSM, FEEDS)

In [ ]:
# employment_grid.explore("pred_relative_employment")

<!-- ### Sample run -->

In [ ]:
sample_origins = employment.sample(20, random_state=10027)

ttm_sample = r5py.TravelTimeMatrix(
    network,
    origins=sample_origins,
    destinations=employment,
    departure=DEPARTURE,
    departure_time_window=datetime.timedelta(minutes=60),
    percentiles=[50],
    transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
    snap_to_network=True,
)
sample_matrix = ttm_sample.rename(columns={"travel_time": "p50"}).filter(
    ["from_id", "to_id", "p50"]
)

sample_matrix.head()

<!-- ### Full run -->

In [4]:
run = False
if run:
    ttm = r5py.TravelTimeMatrix(
        network,
        origins=employment,
        destinations=employment,
        departure=DEPARTURE,
        departure_time_window=datetime.timedelta(minutes=60),
        percentiles=[50],
        transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
        snap_to_network=True,
    )
    matrix = ttm.rename(columns={"travel_time": "p50"})[["from_id", "to_id", "p50"]]
    matrix.to_parquet(OUT / "accessibility_grid_wed_08.parquet", index=False)

    n_pairs = len(matrix)
    n_null = matrix["p50"].isna().sum()
    n_origins = matrix["from_id"].nunique()
    n_dest = matrix["to_id"].nunique()
    print(
        f"{n_pairs} origin-destination pairs, {n_origins} origins, {n_dest} "
        f"destinations, {n_null} null p50 ({100 * n_null / n_pairs:.1f}%)"
    )
else:
    matrix = pd.read_parquet(OUT / "accessibility_grid_wed_08.parquet")

<!-- ## % of jobs reachable within 45 minutes, per grid cell -->

In [5]:
THRESHOLD = 45

cumulative_opportunities = (
    matrix.query("p50 <= @THRESHOLD")
    .merge(employment, left_on="to_id", right_on="id")
    .drop(columns="id")
    .groupby(["from_id"])["pred_relative_employment"]
    .sum()
    .reset_index()
    .rename(columns={"pred_relative_employment": f"pct_jobs_reachable_{THRESHOLD}min"})
)

cumulative_opportunities.head()

,from_id,pct_jobs_reachable_45min
0,montevideo_100091,32.491320
1,montevideo_100092,49.714075
2,montevideo_100093,64.900842
3,montevideo_100094,30.411480
4,montevideo_100095,6.142821


In [6]:
grid_scores = (
    cumulative_opportunities.merge(
        employment_grid.filter(["grid_id", "geometry"]),
        left_on="from_id",
        right_on="grid_id",
    )
    .drop(columns="from_id")
    .pipe(lambda df: gpd.GeoDataFrame(df, geometry="geometry"))
)

In [7]:
grid_scores.explore(
    column="pct_jobs_reachable_45min",
    cmap="YlOrRd",
    legend=True,
    tooltip=["grid_id", "pct_jobs_reachable_45min"],
    tiles="CartoDB positron",
)
